# Kaggle runner — Sanity Y10 (và P0, chỉ khi được yêu cầu)

**Survey v3 (QĐ15, 26/10):** P0 / bài toán giai đoạn không còn thuộc survey; `RUN_P0 = False` mặc định. Y10 vẫn bắt buộc.


Notebook này dùng đúng cấu trúc input đang có: `/kaggle/input/experiment-data-perm/<dataset>/`. Nó chạy theo thứ tự: **(1) Y10** trên `tcga_normal_vs_tumor_sanity`, sau đó **(2) P0** trên ba cohort stage (`GSE68465`, `TCGA-LUAD`, `TCGA-LUSC`), rồi hợp nhất quyết định P0.

Bật **Internet** trong Kaggle Notebook Settings trước khi Run All: mã được clone từ nhánh GitHub `survey-v2-code`, để `config.json` có `commit_hash` và `traceable=true`. Dữ liệu input chỉ-đọc được symlink vào `/kaggle/working/kltn_data/experiment-data-perm`; kết quả chỉ ghi vào `/kaggle/working/kltn_data/results/`.

Cấu hình mặc định là lượt chính thức: Y10 5×5 CV; P0 1.000 permutation + 1.000 random-k, `balanced`, 3 pipeline × 5 mức k × 3 dataset. Có resume an toàn: đính kèm output P0 của lần Kaggle trước làm Dataset, điền `RESUME_P0_DIR`, rồi chạy lại.

In [ ]:
from pathlib import Path

# ===== CẤU HÌNH DUY NHẤT CẦN XEM TRƯỚC KHI RUN ALL =====
INPUT_DATASET = Path('/kaggle/input/experiment-data-perm')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
N_JOBS = 4                 # đặt <= số CPU Kaggle đang cấp
N_PERM = 1000              # lượt chính thức; không giảm nếu muốn kết quả trích dẫn
N_RANDOM = 1000            # lượt chính thức; không giảm nếu muốn kết quả trích dẫn
TIME_BUDGET_H = 11.0       # chừa thời gian cho Kaggle đóng phiên
RUN_P0 = False   # survey v3 (QĐ15): P0 (bài toán giai đoạn) không còn thuộc survey; chỉ bật nếu GVHD yêu cầu cho phụ lục khoá luận

# Nếu tiếp tục một lượt P0 dở dang: thư mục chứa <dataset>/summary.csv từ Kaggle output trước.
# Ví dụ: Path('/kaggle/input/p0-part-1/perm_test_v3'); để None khi chạy mới.
RESUME_P0_DIR = None

STAGE_DATASETS = [
    'gse68465_stage_early_late',
    'tcga_luad_stage_early_late',
    'tcga_lusc_stage_early_late',
]
SANITY_DATASET = 'tcga_normal_vs_tumor_sanity'
REQUIRED_DATASETS = [*STAGE_DATASETS, SANITY_DATASET]


In [ ]:
# 1. Kiểm tra layout input đúng như Data Explorer trước khi cài/chạy bất kỳ thứ gì.
import json
import pandas as pd

if not INPUT_DATASET.is_dir():
    raise FileNotFoundError(f'Không thấy {INPUT_DATASET}. Hãy sửa INPUT_DATASET theo slug Dataset Kaggle.')

required_meta = {'sample_id', 'patient_id', 'class', 'label'}
validation_rows = []
for dataset in REQUIRED_DATASETS:
    root = INPUT_DATASET / dataset
    missing = [str(p.relative_to(INPUT_DATASET)) for p in [root / 'expression.h5', root / 'metadata.csv', root / 'perm_folds.csv'] if not p.is_file()]
    for fold in range(1, 6):
        for split in ('train', 'val', 'test'):
            path = root / 'folds' / f'fold_{fold}' / f'{split}.h5'
            if not path.is_file():
                missing.append(str(path.relative_to(INPUT_DATASET)))
    if missing:
        raise FileNotFoundError('Thiếu file bắt buộc:\n' + '\n'.join(missing))

    meta = pd.read_csv(root / 'metadata.csv')
    folds = pd.read_csv(root / 'perm_folds.csv')
    if required_meta - set(meta.columns):
        raise ValueError(f'{dataset}: metadata.csv thiếu cột {required_meta - set(meta.columns)}')
    if meta['sample_id'].isna().any() or meta['patient_id'].isna().any() or meta['label'].isna().any():
        raise ValueError(f'{dataset}: có sample_id/patient_id/label bị thiếu')
    if not set(meta['label'].unique()) <= {0, 1}:
        raise ValueError(f'{dataset}: label không thuộc {{0, 1}}')
    if meta['sample_id'].duplicated().any() or not meta['sample_id'].astype(str).isin(folds['sample_id'].astype(str)).all():
        raise ValueError(f'{dataset}: sample_id trong metadata/perm_folds không khớp')
    validation_rows.append({'dataset': dataset, 'n_samples': len(meta), 'n_patients': meta['patient_id'].nunique(),
                            'n_label_0': int((meta['label'] == 0).sum()), 'n_label_1': int((meta['label'] == 1).sum())})

validation = pd.DataFrame(validation_rows)
display(validation)
assert set(validation['dataset']) == set(REQUIRED_DATASETS)
print('INPUT_LAYOUT_OK')


In [ ]:
# 2. Clone mã đã push + cài dependencies. Internet trong Kaggle phải được bật.
import os
import shutil
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)

commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
remote_contains = subprocess.check_output(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert not status, f'Repo bẩn trước khi chạy: {status}'
assert remote_contains, 'HEAD chưa nằm trên remote; không chạy kết quả có provenance'
print(f'CODE_OK: {commit_hash} | {remote_contains}')


In [ ]:
# 3. Cấp đường dẫn dữ liệu cho code. Không copy 1.6 GB input: symlink chỉ-đọc là đủ vì scripts chỉ đọc data.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
linked_data = WORK_ROOT / 'experiment-data-perm'
if linked_data.exists() or linked_data.is_symlink():
    if not linked_data.is_symlink() or linked_data.resolve() != INPUT_DATASET.resolve():
        raise RuntimeError(f'{linked_data} đã tồn tại nhưng không phải symlink an toàn đến input; hãy dùng Kaggle session mới.')
else:
    linked_data.symlink_to(INPUT_DATASET, target_is_directory=True)

os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
assert (Path(os.environ['KLTN_DATA_ROOT']) / 'experiment-data-perm' / STAGE_DATASETS[0]).is_dir()
print('KLTN_DATA_ROOT =', os.environ['KLTN_DATA_ROOT'])


In [ ]:
# 4. Y10: phải đạt AUROC trung bình >= 0.99 trước khi tin bất kỳ kết quả P0 nào.
sanity_cmd = [sys.executable, 'scripts/run_sanity_auroc.py']
subprocess.run(sanity_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)

sanity_path = WORK_ROOT / 'results' / 'survey_v2' / 'sanity' / 'sanity_summary.csv'
sanity = pd.read_csv(sanity_path)
display(sanity)
assert len(sanity) == 3 and sanity['pass'].all(), 'Y10 không đạt; P0 không được diễn giải'
print('Y10_OK')


In [ ]:
# 5. P0: 3 cohort stage, 3 pipeline (ftest_lr, svm_rfe, pam), k=10/50/100/200/500.
if RUN_P0:
    p0_cmd = [sys.executable, 'scripts/run_p0_perm_v3.py', '--datasets', *STAGE_DATASETS,
              '--class-weights', 'balanced', '--n-perm', str(N_PERM), '--n-random', str(N_RANDOM),
              '--n-jobs', str(N_JOBS), '--time-budget-h', str(TIME_BUDGET_H)]
    if RESUME_P0_DIR is not None:
        if not RESUME_P0_DIR.is_dir():
            raise FileNotFoundError(f'RESUME_P0_DIR không tồn tại: {RESUME_P0_DIR}')
        p0_cmd += ['--resume-dirs', str(RESUME_P0_DIR)]
    subprocess.run(p0_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)

    summary_paths = [WORK_ROOT / 'results' / 'survey_v2' / 'P0' / 'perm_test_v3' / d / 'summary.csv' for d in STAGE_DATASETS]
    if not all(p.is_file() for p in summary_paths):
        raise RuntimeError('P0 chưa hoàn tất đủ 3 cohort. Hãy Save Version, đính output làm Dataset và đặt RESUME_P0_DIR.')
    aggregate_cmd = [sys.executable, 'scripts/aggregate_p0_decision.py', '--summaries', *map(str, summary_paths)]
    subprocess.run(aggregate_cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# 6. Chỉ chấp nhận output đầy đủ và truy vết được; sau đó nén để Download/Save Version.
import zipfile

RESULTS = WORK_ROOT / 'results' / 'survey_v2'
config_paths = [RESULTS / 'sanity' / 'config.json']
if RUN_P0:
    config_paths += [RESULTS / 'P0' / 'perm_test_v3' / d / 'config.json' for d in STAGE_DATASETS]
for config_path in config_paths:
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    assert cfg.get('traceable') is True, f'Không traceable: {config_path}'
    assert cfg.get('commit_hash') == commit_hash, f'Commit mismatch: {config_path}'

if RUN_P0:
    decision = pd.read_csv(RESULTS / 'P0' / 'p0_decision.csv')
    display(decision)
    assert set(decision['dataset']) == set(STAGE_DATASETS)
    assert (decision['n_perm'] == N_PERM).all() and (decision['n_random'] == N_RANDOM).all()
    assert decision['family_complete'].all(), 'P0 chưa đủ family; tiếp tục resume, không diễn giải p-value'

archive = Path('/kaggle/working/survey_v2_y10_p0_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in RESULTS.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(WORK_ROOT))
print('RESULTS_OK:', RESULTS)
print('DOWNLOAD:', archive)
print('Kaggle: Save Version để giữ output; nếu P0 bị hết giờ, đính output ở lượt sau và đặt RESUME_P0_DIR.')
